# DT401 Workshop 1 — Exploring Database Structure with SQL

## Learning Objectives

By the end of this notebook, you will be able to:
- Query the `sqlite_master` table to list all tables in a database
- Use `pragma_table_info` to retrieve column names, data types, and primary keys
- Combine `sqlite_master` and `pragma_table_info` using a Common Table Expression (CTE) to generate a complete schema summary
- Understand when to use programmatic schema exploration instead of (or alongside) an ERD


---

## Context Recap

When working with unfamiliar databases, you need to understand their structure before you can write effective queries. Entity Relationship Diagrams (ERDs) provide a visual overview of tables and relationships, but they are not always available — and even when they are, you may need to verify exact column names, data types, or primary keys programmatically.

This notebook shows you how to explore a SQLite database's structure using SQL itself. You will query the database's metadata tables to extract information about what tables exist and what columns they contain. These techniques are particularly valuable when working with legacy systems, third-party databases, or any situation where documentation is incomplete or out of date.

**Prerequisites:** This is a supplementary notebook that can be used alongside any of the main Workshop 1 notebooks. Familiarity with basic SQL queries from [1 — Selecting Data](1_selecting_data.ipynb) and the structure of an [ERD](ERD.ipynb) is recommended.

---

## Setup

Run this cell before starting. If your kernel restarts, run it again.

In [ ]:
# Setup: load the SQL extension and connect to the pagila database
%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False
%config SqlMagic.autopolars = False
%config SqlMagic.displaycon = False

%sql sqlite:///data/pagila.db

## Part 1: When You Don't Have an ERD

In practice, you will often need to explore a database whose structure is not documented. A colleague might hand you access to a database and ask you to analyse it — no diagram, no data dictionary, no context. Knowing how to extract schema information directly from the database is a core skill for any data professional.

SQLite stores metadata about its own structure in special system tables and functions. Two are particularly useful: the `sqlite_master` table, which lists every object in the database, and the `pragma_table_info` function, which describes the columns of any given table. Both are SQLite-specific — other database systems (PostgreSQL, MySQL, SQL Server) have their own equivalents, but the principle is the same.

**Reinforcement Resource**

[The SQLite Schema Table — sqlite.org](https://www.sqlite.org/schematab.html) — official documentation for `sqlite_master`, explaining what it stores and how to query it.

*Read time: 5 minutes*

### Worked Example: Listing All Tables

**What you'll see:** A query that retrieves every table name from `sqlite_master`. This is typically your first step when exploring an unfamiliar SQLite database.

In [ ]:
%%sql
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table';

The `sqlite_master` table records every object in the database — tables, views, indexes, and triggers. Filtering by `type = 'table'` isolates just the tables. The result gives you a complete inventory of what the database contains before you write a single analytical query.

**Observation task:** Run the query above. How many tables does the pagila database contain?

<details>
<summary>Solution</summary>

The pagila database contains 14 tables: `actor`, `address`, `category`, `city`, `country`, `customer`, `film`, `film_actor`, `film_category`, `inventory`, `language`, `payment`, `rental`, and `store`.

</details>

## Part 2: Exploring Column Structure

Knowing which tables exist is useful, but you also need to know what columns each table contains — their names, data types, and which column (if any) serves as the primary key. The `pragma_table_info` function provides exactly this. You can call it on any table name and it returns one row per column.

The query below uses a Common Table Expression (CTE) to first retrieve all table names from `sqlite_master`, then joins that list to `pragma_table_info` to pull column details for every table in one pass. This approach is more efficient than calling `pragma_table_info` separately for each table.

**Reinforcement Resource**

[PRAGMA table_info — sqlite.org](https://www.sqlite.org/pragma.html#pragma_table_info) — official documentation describing every column returned by `pragma_table_info`, including the `pk` field that identifies primary keys.

*Read time: 3 minutes*

### Worked Example: Full Schema Summary

**What you'll see:** A CTE-based query that combines `sqlite_master` with `pragma_table_info` to produce a single table listing every column in the database, along with its data type and whether it is a primary key.

In [ ]:
%%sql
WITH table_info AS (
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
)
SELECT table_name, name AS column_name, type AS data_type, pk AS primary_key
FROM table_info
JOIN pragma_table_info(table_name)
ORDER BY table_name, column_name;

The `pk` column in the output contains a `1` for any column that is (part of) the primary key and a `0` otherwise. Scanning this output lets you quickly identify the primary key of each table and the data types in use — both of which are important when writing JOIN queries or filtering by date and numeric fields.

**Observation task:** Run the query above. Which table has the most columns? How does that compare to the ERD?

<details>
<summary>Solution</summary>

The `film` table has the most columns — 13 in total. This reflects the amount of detail stored about each film (title, description, release year, language, rental duration, rental rate, length, replacement cost, rating, and special features). Checking this against the ERD confirms the match: the ERD shows `film` as one of the most connected and data-rich tables in the schema.

</details>

## When to Use This Approach

ERDs and programmatic schema queries serve different purposes. An ERD shows you the relationships between tables — which is essential for writing JOIN queries. Programmatic schema queries show you exact column names and data types — which is essential for writing correct filtering and aggregation queries, and for debugging type errors.

In practice, you will use both. Start with an ERD if one is available; use `sqlite_master` and `pragma_table_info` to fill in the details or to explore databases where no ERD exists.

For a Python-based approach to database inspection using SQLAlchemy, see [Notebook 7](7_sql_interactions.ipynb). For a visual representation of the pagila database structure, see the [ERD notebook](ERD.ipynb).

---

## Summary

**You have completed the Exploring Database Structure with SQL notebook.** You can now:
- Retrieve all table names from a SQLite database using `sqlite_master`
- Extract column names, data types, and primary key information using a CTE with `pragma_table_info`
- Decide when to use programmatic schema exploration alongside or instead of an ERD

**Next:** Return to the [Workshop README](README.ipynb) to choose your next notebook, or continue with [Notebook 7](7_sql_interactions.ipynb) to see how Python and SQLAlchemy can inspect database structure using a different approach.

**Action items:**
1. Save this notebook (Ctrl+S)
2. Commit to Git: `"Complete Workshop 1 SQLite table identification"`

---

**Notebook Version:** 1.0
**Last Updated:** March 2026